# Explorador de imputación

Selecciona Ishii o MMC2 en la primera celda y ejecuta **Ejecutar todo**.
El notebook presenta dos partes: evaluación de mediciones ocultadas y candidatos reales.
Lee resultados existentes y muestra gráficos; no entrena modelos ni exporta archivos.

Guárdalo junto a `ejecutar_imputacion.py`. Instala las dependencias con
`python -m pip install -r requirements.txt` y selecciona ese Python como kernel.
Para generar resultados usa `python ejecutar_imputacion.py --dataset ishii --output resultados/ishii`
(o MMC2 con su carpeta). El ejecutor exige una carpeta de salida nueva.


In [ ]:
# 1. CONFIGURACIÓN: modifica estas opciones y ejecuta todo.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

DATASET = "ishii"  # "ishii" o "mmc2"
METABOLITO = "fum[c]"  # fum[c], pep[c], s7p[c], succ[c]
EXPERIMENTO = "perturbation"  # Opción de visualización.
POLITICA = "available"  # available o supported_plus_baseline
SOLO_CASOS_COMUNES = (
    True  # comparar sobre los mismos casos aceptados por los tres
)
PATRON = None  # para real_patterns: None = todos; o "P01", "P02", etc.
RUTA_PROYECTO = (
    None  # opcional: r"C:\Users\TuUsuario\...\IPre_Normalizacion_Imputacion"
)

METHODS = ["SimpleImputer", "KNNImputer", "IterativeImputer"]
NAMES = {
    "SimpleImputer": "Mediana",
    "KNNImputer": "KNN",
    "IterativeImputer": "Iterative",
}
COLORS = {
    "SimpleImputer": "#777777",
    "KNNImputer": "#16869b",
    "IterativeImputer": "#b5548c",
}
LABELS = {
    "fum[c]": "Fumarato",
    "pep[c]": "PEP",
    "s7p[c]": "s7p",
    "succ[c]": "Succinato",
}
assert DATASET in ("ishii", "mmc2")
assert METABOLITO in LABELS
assert POLITICA in ("available", "supported_plus_baseline")

if RUTA_PROYECTO:
    ROOT = Path(RUTA_PROYECTO).expanduser().resolve()
else:
    candidates = [
        Path.cwd(),
        Path.cwd() / "IPre_Normalizacion_Imputacion",
        Path.cwd().parent,
    ]
    ROOT = next(
        (
            p
            for p in candidates
            if (p / "normalizacion").is_dir()
            and (p / "ejecutar_imputacion.py").is_file()
        ),
        None,
    )
    if ROOT is None:
        raise FileNotFoundError(
            "No encuentro el proyecto. Define RUTA_PROYECTO."
        )
sys.path.insert(0, str(ROOT))
RES = ROOT / "resultados" / DATASET
plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)
print(f"Proyecto: {ROOT}\nBase seleccionada: {DATASET}\nResultados: {RES}")


In [ ]:
# Leer una única base y sus resultados ya calculados.
def leer_csv(archivo, index_col=None):
    if not archivo.is_file():
        raise FileNotFoundError(
            f"Falta {archivo}. Ejecuta primero el análisis de {DATASET}."
        )
    return pd.read_csv(archivo, index_col=index_col)


from apoyo import describir  # noqa: E402

original = leer_csv(
    ROOT / "normalizacion" / f"matriz_{DATASET}_mM.csv", index_col=0
)
pred = leer_csv(RES / "validacion/predicciones.csv")
casos = leer_csv(RES / "validacion/casos_comunes.csv")
reales = leer_csv(RES / "imputacion/candidatos_y_soporte.csv")
casillas = leer_csv(RES / "imputacion/casillas_comunes.csv")
estadisticas = leer_csv(RES / "imputacion/estadisticas_antes_despues.csv")
mascaras = json.loads(
    (RES / "validacion/mascaras.json").read_text(encoding="utf-8")
)
manifest = json.loads((RES / "manifest.json").read_text(encoding="utf-8"))
assert manifest["dataset"] == DATASET
print(f"{len(original)} muestras y {len(original.columns)} metabolitos.")
print("Pruebas disponibles:", ", ".join(pred.experiment.unique()))
if DATASET == "mmc2":
    display(
        Markdown(
            (
                "**MMC2:** no hay prueba por serie equivalente a "
                "Ishii. Glucosa se excluye como consulta "
                "evaluada porque su concentración está revelada "
                "en la referencia de normalización."
            )
        )
    )


## Datos originales: todavía no hay imputación

Cada fila representa una muestra o condición; cada columna, un metabolito. `NaN` significa ausencia. Las posiciones del gráfico identifican muestras, **no representan una serie temporal**. Todas las concentraciones mostradas están en mM.


In [ ]:
display(original.round(5))
resumen_original = pd.DataFrame(
    {
        "observados": original.count(),
        "faltantes": original.isna().sum(),
        "media_mM": original.mean(),
        "mediana_mM": original.median(),
        "SD_mM": original.std(ddof=1),
    }
)
display(resumen_original.round(5))
fig, ax = plt.subplots(figsize=(12, 3.5))
x = np.arange(len(original))
ax.scatter(x, original[METABOLITO], color="#202b38", label="Medición original")
ax.set_xticks(x, original.index, rotation=90)
ax.set_ylabel("Concentración (mM)")
ax.set_title(f"{DATASET.upper()} — {LABELS[METABOLITO]} antes de imputar")
ax.grid(axis="y", alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()


## Sección 1 — Evaluación: valor conocido ocultado frente a valor imputado

Aquí **sí conocemos la verdad**: es una medición original que se ocultó al modelo. El error compara esa medición con su estimación. Los métodos reciben las mismas máscaras.

- `available`: incluye todos los candidatos finitos, incluso sustituciones generales o ajustes no convergentes.
- `supported_plus_baseline`: acepta solo predicciones con soporte de KNN e Iterative y conserva la mediana como referencia general. Esa excepción no convierte la mediana en una predicción individual con soporte.
- `SOLO_CASOS_COMUNES=True`: compara únicamente los casos aceptados por los tres métodos.

La tabla de cobertura siempre muestra los casos propios y comunes. Si la intersección está vacía, no se inventan comparaciones. En patrones reales una muestra puede aparecer en varios ensayos; esos puntos no son observaciones independientes.


In [ ]:
# Mostrar qué oculta cada patrón real.
display(
    pd.DataFrame(
        [
            {
                "patron": p["pattern_id"],
                "variables_ocultas": ", ".join(p["hidden"]),
                "objetivos_evaluados": ", ".join(p["score"]),
                "muestras_origen": ", ".join(p["source_samples"]),
            }
            for p in mascaras["patterns"]
        ]
    )
)

if EXPERIMENTO not in set(pred.experiment):
    raise ValueError(
        f"Prueba no disponible: {EXPERIMENTO}. "
        f"Opciones: {list(pred.experiment.unique())}"
    )
base = pred[
    (pred.experiment == EXPERIMENTO) & (pred.metabolite_id == METABOLITO)
].copy()
if EXPERIMENTO == "real_patterns" and PATRON is not None:
    trials = {
        t["trial_id"] for t in mascaras["trials"] if t["pattern_id"] == PATRON
    }
    if not trials:
        raise ValueError(f"Patrón sin ensayos elegibles: {PATRON}.")
    base = base[base.trial_id.isin(trials)].copy()
criterios = casos[casos.policy == POLITICA]
base = base.merge(
    criterios[["method", "case_id", "responded", "common_methods"]],
    on=["method", "case_id"],
    validate="one_to_one",
)
base["aceptado"] = base.responded
comunes = set(base.loc[base.common_methods, "case_id"])
comparacion = base[base.aceptado].copy()
if SOLO_CASOS_COMUNES:
    comparacion = comparacion[comparacion.common_methods].copy()
rows = []
for method in METHODS:
    propios = base[(base.method == method) & base.aceptado]
    seleccion = comparacion[comparacion.method == method]
    attempts = int((base.method == method).sum())
    rows.append(
        {
            "metodo": NAMES[method],
            "intentos": attempts,
            "respuestas_propias": len(propios),
            "cobertura_propia_%": (
                100 * len(propios) / attempts if attempts else np.nan
            ),
            "casos_comunes": len(comunes),
            "casos_graficados": len(seleccion),
            "MAE_mM_seleccion": seleccion.absolute_error_mM.mean(),
            "MAE_R_seleccion": seleccion.absolute_error_R.mean(),
        }
    )
display(pd.DataFrame(rows).round(6))
if comparacion.empty:
    display(
        Markdown(
            (
                "**No hay casos aceptados para esta selección.** "
                "Puedes explorar `available` o desactivar casos "
                "comunes, indicando que las comparaciones "
                "podrían usar distintas celdas."
            )
        )
    )
else:
    display(
        comparacion[
            [
                "sample_id",
                "trial_id",
                "method",
                "truth_mM",
                "estimate_mM",
                "absolute_error_mM",
                "support",
                "mechanism",
            ]
        ].round(6)
    )


In [ ]:
# Dos perspectivas de los mismos casos: comparación pareada y dispersión
# real vs estimado.
if not comparacion.empty:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    maximo = max(
        comparacion.truth_mM.max(), comparacion.estimate_mM.max(), 0.001
    )
    for j, method in enumerate(METHODS):
        d = comparacion[comparacion.method == method].sort_values("case_id")
        ax = axes[0, j]
        xx = np.arange(len(d))
        ax.vlines(xx, d.truth_mM, d.estimate_mM, color="#bbbbbb", lw=1)
        ax.scatter(
            xx,
            d.truth_mM,
            color="#202b38",
            s=28,
            label="Medido, luego ocultado",
        )
        ax.scatter(
            xx,
            d.estimate_mM,
            color=COLORS[method],
            marker="x",
            s=36,
            label="Imputado",
        )
        labels = [
            (
                f"{r.sample_id}/{r.trial_id.split(':')[-1]}"
                if EXPERIMENTO == "real_patterns"
                else r.sample_id
            )
            for r in d.itertuples()
        ]
        step = max(1, int(np.ceil(len(d) / 18)))
        ax.set_xticks(xx[::step], labels[::step], rotation=90, fontsize=8)
        ax.set_title(f"{NAMES[method]} — n={len(d)}")
        ax.set_ylabel("Concentración (mM)")
        ax.set_ylim(-0.03 * maximo, 1.08 * maximo)
        ax.legend(fontsize=8)
        ax = axes[1, j]
        ax.scatter(d.truth_mM, d.estimate_mM, color=COLORS[method], alpha=0.75)
        ax.plot([0, maximo], [0, maximo], "--", color="#777777")
        ax.set(
            xlabel="Medición original ocultada (mM)",
            ylabel="Estimación (mM)",
            xlim=(-0.03 * maximo, 1.08 * maximo),
            ylim=(-0.03 * maximo, 1.08 * maximo),
        )
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)
    fig.suptitle(
        f"Validación — {DATASET.upper()}, {LABELS[METABOLITO]}\n"
        f"{EXPERIMENTO}; {POLITICA}; casos comunes: {SOLO_CASOS_COMUNES}"
    )
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()


In [ ]:
# Estadísticas de las verdades ocultadas y sus estimaciones SOBRE LOS
# CASOS EVALUADOS.
# No se confunden con estadísticas de toda la cohorte ni de los huecos reales.
resumen_validacion = []
for method in METHODS:
    d = comparacion[comparacion.method == method]
    for tipo, columna in [
        ("Mediciones ocultadas", "truth_mM"),
        ("Estimaciones", "estimate_mM"),
    ]:
        resumen_validacion.append(
            {
                "metodo": NAMES[method],
                "datos": tipo,
                "n": d[columna].count(),
                **describir(d[columna]),
            }
        )
display(pd.DataFrame(resumen_validacion).round(6))


## Sección 2 — Candidatos reales: completar huecos que no tenían medición

Aquí **no conocemos el valor verdadero**: no es posible calcular su MAE. Se conserva cada medición original y solo se completan los candidatos aceptados según la política seleccionada.

El gráfico distingue valores medidos de imputados. Las estadísticas se calculan sobre todas las muestras de la base seleccionada, omitiendo los valores pendientes. Con `SOLO_CASOS_COMUNES=True`, cada método completa exactamente las mismas celdas aceptadas por los tres. Una intersección vacía deja las estadísticas originales sin cambios: no demuestra que los métodos sean equivalentes.


In [ ]:
# Las decisiones de aceptación ya están registradas por el ejecutor.
criterios = casillas[casillas.policy == POLITICA]
r = reales.merge(
    criterios[
        ["method", "sample_id", "metabolite_id", "selected", "common_methods"]
    ],
    on=["method", "sample_id", "metabolite_id"],
    validate="one_to_one",
)
r["aceptado"] = r.selected
if SOLO_CASOS_COMUNES:
    r["aceptado"] = r.common_methods

matrices = {}
for method in METHODS:
    matriz = original.copy()
    for registro in r[(r.method == method) & r.aceptado].itertuples():
        assert pd.isna(
            original.loc[registro.sample_id, registro.metabolite_id]
        )
        matriz.loc[registro.sample_id, registro.metabolite_id] = (
            registro.estimate_mM
        )
    matrices[method] = matriz

print(f"Candidatos en los cuatro objetivos: {len(reales) // len(METHODS)}")
display(
    r[r.metabolite_id == METABOLITO][
        [
            "sample_id",
            "method",
            "estimate_mM",
            "aceptado",
            "support",
            "mechanism",
        ]
    ].round(6)
)
if not original[METABOLITO].isna().any():
    display(
        Markdown(
            (
                "**Este metabolito no tiene huecos reales en la "
                "base seleccionada.**"
            )
        )
    )


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
x = np.arange(len(original))
observed = original[METABOLITO].notna()
for ax, method in zip(axes, METHODS):
    column = matrices[method][METABOLITO]
    imputed = ~observed & column.notna()
    pending = ~observed & column.isna()
    ax.scatter(
        x[observed],
        original.loc[observed, METABOLITO],
        color="#202b38",
        s=26,
        label="Medido",
    )
    ax.scatter(
        x[imputed],
        column[imputed],
        color=COLORS[method],
        marker="x",
        s=48,
        label="Imputado",
    )
    ax.set_xticks(x, original.index, rotation=90, fontsize=7)
    ax.set_title(
        f"{NAMES[method]}: {int(imputed.sum())} imputados; "
        f"{int(pending.sum())} pendientes"
    )
    ax.set_ylabel("Concentración (mM)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.2)
fig.suptitle(
    f"Huecos reales — {DATASET.upper()}, {LABELS[METABOLITO]}\n"
    f"{POLITICA}; casillas comunes: {SOLO_CASOS_COMUNES}"
)
plt.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()

# Leer las estadísticas que corresponden a las casillas mostradas.
alcance = "common_methods" if SOLO_CASOS_COMUNES else "own"
resumen = estadisticas[
    (estadisticas.policy == POLITICA)
    & (estadisticas.scope == alcance)
    & (estadisticas.metabolite_id == METABOLITO)
]
display(
    resumen[
        [
            "method",
            "stage",
            "total_samples",
            "observed",
            "imputed",
            "pending",
            "mean_mM",
            "median_mM",
            "SD_mM",
            "IQR_mM",
            "CV_pct",
            "CV_status",
        ]
    ].round(6)
)


## Cómo interpretar las comparaciones

- Menor MAE significa menor error promedio **en las mediciones ocultadas evaluadas**; no garantiza exactitud en los huecos reales.
- La cobertura numérica incluye sustituciones generales. La convergencia y el soporte se revisan por separado.
- Una caída de SD o CV después de imputar puede deberse a rellenar con valores centrales; no demuestra mejor precisión.
- Las estadísticas tratan los imputados como estimaciones puntuales, sin incorporar incertidumbre de imputación o de la referencia.
- La escala del análisis es lineal, sin logaritmos. Las concentraciones y errores se muestran en mM; la tabla de validación también incluye MAE en ratios.
- Para cambiar de base, modifica `DATASET` y ejecuta todo. No se combinan Ishii y MMC2.
